# Session 5 — Monitoring Model Explainability and Data Drift using Evidently AI

**Goal:** take a trained classifier, feed it a "production" slice of data whose
distribution has quietly shifted away from what it was trained on, and use
**Evidently AI** to detect that shift, quantify it per column, and check whether
the model's explanations have moved along with it.

## What this session automates

Sessions 1-4 all end in the same place: a trained model with a good validation
score. That score is a snapshot taken on data drawn from the training distribution.
Production is not that distribution — customer mixes change, campaigns target new
segments, upstream systems start emitting a new category value. Nothing inside the
model notices; it keeps returning confident predictions on inputs it has never
really seen.

Evidently AI automates the "is the input still the input?" check. Hand it two
DataFrames — a **reference** (what the model trained on) and a **current** (what
production is sending) — and it picks an appropriate statistical test per column,
runs it, and gives you both a readable report and a machine-readable dict you can
gate a pipeline on. Session 17 wires that signal into an automatic retraining
trigger; this session is about understanding the signal first.

## The dataset

This session uses the UCI **Bank Marketing** dataset (`id=222`) — 45,211 records
from a Portuguese bank's direct-marketing phone campaigns, with 16 features
(client age, job, marital status, balance, loan status, contact channel, campaign
history) and a binary target `y`: did the client subscribe to a term deposit?

It fits this session unusually well because the drift is *already in the data*.
The records are ordered chronologically across a multi-year campaign, and the bank
changed who it called and how it called them over that period — `month`, `contact`,
`duration`, and `poutcome` all shift substantially between early and late records.
We don't have to invent a synthetic shift; we just split on time and look.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe*
says exactly what to look at in that cell's output; *Infer* says what conclusion
that output should lead you to, and what it would mean if you saw something
different instead. Drift monitoring is full of numbers that are easy to misread —
a p-value and a distance score both arrive in a column called `drift_score`, and
they point in opposite directions — so read the Infer notes rather than skimming
for a green checkmark.

## Prerequisites

Everything here runs locally — no cloud account needed. Evidently's API changed
substantially at version 0.7, so pin the version this notebook was written
against:

```bash
pip install "evidently==0.4.40" scikit-learn pandas ucimlrepo
```

The Step 9 callout covers what breaks if you install a newer version by accident,
since that is by far the most common failure people hit with this library.

## Step 1 — Fetch the dataset

Fetching directly from the UCI ML Repository keeps this notebook runnable by
anyone, rather than depending on a CSV already sitting on your machine.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

bank = fetch_ucirepo(id=222)
df = pd.concat([bank.data.features, bank.data.targets], axis=1)
df.columns = [c.strip() for c in df.columns]

print(f"{len(df)} rows, {len(df.columns)} columns")
print("Target distribution:")
print(df["y"].value_counts(normalize=True).round(4))
df.head()

**Observe:** the printed shape (`45211 rows, 17 columns` — 16 features plus
the target `y`) and the target distribution: roughly **88.3% `no` / 11.7% `yes`**.

**Infer:** the imbalance shapes everything downstream. A model that predicts `no`
for every client scores 88% accuracy, so accuracy is useless as a health metric
here — Step 4 uses ROC AUC instead. It also matters for drift: on an imbalanced
target a small absolute change in the positive rate is a large *relative* change,
which is why Step 7 treats target drift separately. If your row count differs from
45,211, `fetch_ucirepo` returned the smaller 4,521-row `bank.csv` sample and every
drift statistic below will be far noisier than described.

## Step 2 — Split into a reference slice and a "production" slice

The records are in campaign order, so an index-based split is a time split. We
take the **first 60%** as the reference (the historical data a model would have
been trained on) and the **last 25%** as the current production window, leaving a
gap in between so the two windows aren't adjacent — mimicking a model that was
trained a while ago and has been serving since.

In [ ]:
n = len(df)
reference = df.iloc[: int(0.60 * n)].reset_index(drop=True)
current = df.iloc[int(0.75 * n) :].reset_index(drop=True)

print(f"Reference window: {len(reference)} rows")
print(f"Current window  : {len(current)} rows")

for col in ["age", "duration", "campaign"]:
    print(f"{col:<10} ref mean {reference[col].mean():8.2f}   cur mean {current[col].mean():8.2f}")

print()
print("contact channel mix -- reference:")
print(reference["contact"].value_counts(normalize=True).round(3))
print("contact channel mix -- current:")
print(current["contact"].value_counts(normalize=True).round(3))

**Observe:** the two row counts (`27126` and `11303`), the three numeric
means, and especially the `contact` mix. In a real run the reference window is
dominated by `unknown` contact channel (**~35%**) while the current window is
almost entirely `cellular` (**~90%+**, with `unknown` down near 2%).

**Infer:** the `contact` shift is the headline finding of this notebook and it is
not noise — the bank moved from an untracked/landline mix to mobile-first calling
partway through the campaign. This is how drift breaks models silently:
`contact=unknown` was genuinely predictive in training (it tracked the older,
lower-yield records), and in production that category has essentially vanished. The
model still holds a weight for it; the branch just never fires. Note `duration` and
`campaign` move noticeably while `age` barely does — a report where *everything*
moves usually means you split on the wrong axis, and one where nothing moves gives
you no signal to test the detector against.

## Step 3 — Train the baseline model on the reference window

Deliberately simple: one-hot encode the categoricals, fit a gradient-boosted tree.
The point of this session is the monitoring, not the model.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

CAT_COLS = ["job", "marital", "education", "default", "housing",
            "loan", "contact", "month", "poutcome"]
NUM_COLS = ["age", "balance", "day_of_week", "duration",
            "campaign", "pdays", "previous"]
FEATURES = NUM_COLS + CAT_COLS


def encode(frame, columns=None):
    X = pd.get_dummies(frame[FEATURES], columns=CAT_COLS)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0)
    return X


X_ref = encode(reference)
y_ref = (reference["y"] == "yes").astype(int)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_ref, y_ref, test_size=0.2, random_state=42, stratify=y_ref
)

model = HistGradientBoostingClassifier(max_iter=200, random_state=42)
model.fit(X_tr, y_tr)

val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
print(f"Validation ROC AUC (reference window): {val_auc:.4f}")
print(f"Encoded feature count: {X_ref.shape[1]}")

**Observe:** the validation ROC AUC — a real run scores around **0.928** —
and the encoded feature count, around **51** columns after one-hot expansion.

**Infer:** 0.928 is healthy, and it's the number a team puts on a model card and
stops thinking about. It was measured on a held-out slice of the *reference*
window — same distribution as training. Step 4 measures the same model on the
production window, and the gap between those two numbers is the entire argument
for monitoring. Note the `reindex` inside `encode()`: one-hot encoding the current
window independently would yield a *different* column set if a category
disappeared, and silently misaligned columns are far nastier than a loud error.

## Step 4 — Score the model on both windows

Before running any drift tooling, establish whether there is actually a problem
worth detecting. This is the ground truth the drift report is trying to predict
*without* labels.

In [ ]:
X_cur = encode(current, columns=X_ref.columns)
y_cur = (current["y"] == "yes").astype(int)

ref_scores = model.predict_proba(X_val)[:, 1]
cur_scores = model.predict_proba(X_cur)[:, 1]

cur_auc = roc_auc_score(y_cur, cur_scores)

print(f"ROC AUC on reference hold-out : {val_auc:.4f}")
print(f"ROC AUC on production window  : {cur_auc:.4f}")
print(f"Degradation                   : {val_auc - cur_auc:+.4f}")
print()
print(f"Mean predicted probability -- reference: {ref_scores.mean():.4f}")
print(f"Mean predicted probability -- current  : {cur_scores.mean():.4f}")
print(f"Actual positive rate       -- current  : {y_cur.mean():.4f}")

**Observe:** the two AUCs and, more importantly, the last three lines. A real
run shows AUC dropping from **0.928 to roughly 0.874**, while the model's mean
predicted probability stays near **0.117** and the *actual* positive rate in the
production window is closer to **0.19**.

**Infer:** two distinct failures. The AUC drop (~0.05) says the *ranking* got
worse — likely subscribers still sort above unlikely ones, just less cleanly. The
gap between mean predicted probability (0.117) and actual positive rate (0.19) says
the model is **systematically under-predicting**: the production population
subscribes at a higher rate than the model has any way of knowing. Miscalibration
is often more damaging than the AUC drop, because anything downstream that
thresholds the raw probability (a call list cut at p > 0.5) will under-select.
Crucially, both were only measurable because we happen to hold labels for the
production window — in reality they'd arrive weeks or months later, which is
exactly why the rest of this notebook works on inputs alone.

## Step 5 — Run Evidently's data drift report

`DataDriftPreset` is a bundle of metrics: it runs a per-column statistical test
plus a dataset-level summary. Evidently picks the test per column automatically
based on type and cardinality — Kolmogorov-Smirnov for numeric columns with enough
rows, chi-squared or Jensen-Shannon distance for categoricals — which is the main
thing it saves you from hand-writing.

`ColumnMapping` tells Evidently which column is the target and which columns are
categorical, so it doesn't have to guess (it guesses badly on integer-coded
categoricals).

In [ ]:
from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

column_mapping = ColumnMapping(
    target="y",
    prediction=None,
    numerical_features=NUM_COLS,
    categorical_features=CAT_COLS,
)

drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(
    reference_data=reference,
    current_data=current,
    column_mapping=column_mapping,
)

drift_report.save_html("bank_drift_report.html")
print("Wrote bank_drift_report.html")
drift_report

**Observe:** the inline rendered report (the `drift_report` object renders as
an interactive HTML widget in Jupyter) — specifically the **Dataset Drift** banner
at the top, which reads something like *"Dataset Drift is detected. 11 out of 16
columns have drifted."* Below it, a table with one row per column, each with a
distribution overlay plot and a **Stat Test** / **Drift Score** pair.

**Infer:** the banner is a *rule*, not a measurement — Evidently declares
dataset-level drift when more than 50% of columns individually drifted. That
threshold is arbitrary: a dataset where one critical feature drifted hard and
fifteen irrelevant ones held steady shows a green banner and still breaks your
model. Step 6 pulls out the per-column numbers so you can apply your own judgment.
If the report renders as a blank white box, that's a Jupyter widget/JS problem, not
an Evidently failure — open the saved `bank_drift_report.html` in a browser.

## Step 6 — Pull the drift numbers out as data

The HTML is for humans. `as_dict()` is what you gate a pipeline on.

In [ ]:
result = drift_report.as_dict()
summary = result["metrics"][0]["result"]

print(f"Columns analysed      : {summary['number_of_columns']}")
print(f"Columns drifted       : {summary['number_of_drifted_columns']}")
print(f"Share of drifted cols : {summary['share_of_drifted_columns']:.3f}")
print(f"Dataset drift flag    : {summary['dataset_drift']}")
print()

by_column = result["metrics"][1]["result"]["drift_by_columns"]
rows = [
    {
        "column": name,
        "stat_test": info["stattest_name"],
        "drift_score": round(info["drift_score"], 5),
        "threshold": info["stattest_threshold"],
        "drifted": info["drift_detected"],
    }
    for name, info in by_column.items()
]
drift_table = pd.DataFrame(rows).sort_values("drifted", ascending=False)
drift_table

**Observe:** the summary block (`16` columns analysed, roughly `11` drifted,
share around `0.688`, `dataset_drift: True`) and then the per-column table. Read
the `stat_test` column alongside `drift_score` — you should see a mix of
`chi-square p_value` (categoricals like `job`, `marital`), `Jensen-Shannon
distance` (high-cardinality categoricals), and `K-S p_value` (numerics like `age`,
`balance`). `contact`, `month`, `poutcome`, `pdays`, and `duration` should all show
`drifted: True`.

**Infer:** **the `drift_score` column does not mean the same thing in every row,
and this is the single most misread number in drift monitoring.** For a p-value
test (`chi-square p_value`, `K-S p_value`) the score is a p-value, the threshold is
0.05, and drift is flagged when the score is **below** it — small means drifted.
For a distance test (`Jensen-Shannon distance`, `Wasserstein distance`, PSI) the
score is a distance, the threshold is typically 0.1, and drift is flagged when the
score is **above** it — large means drifted. Sorting this table by `drift_score` to
find "the worst drift" therefore produces nonsense unless you group by
`stat_test` first. Always read `drift_detected` for the verdict and use
`drift_score` only to compare columns tested the same way.

Second, note the test selection: Evidently switches from chi-squared to
Jensen-Shannon once cardinality or sample size makes the chi-squared p-value
degenerate. With 27,000 reference rows, p-value tests get extremely sensitive — a
shift far too small to matter still returns p < 0.05. That's what a p-value does at
large n, and it's why the next cell measures effect size rather than trusting the
flag.

In [ ]:
# p-values collapse at this sample size, so quantify how big each shift
# actually is, independent of the significance test
from scipy.spatial.distance import jensenshannon
import numpy as np


def category_shift(col):
    ref_p = reference[col].value_counts(normalize=True)
    cur_p = current[col].value_counts(normalize=True)
    idx = ref_p.index.union(cur_p.index)
    return jensenshannon(
        ref_p.reindex(idx, fill_value=1e-9).values,
        cur_p.reindex(idx, fill_value=1e-9).values,
        base=2,
    )


effects = pd.DataFrame(
    [{"column": c, "js_distance": round(float(category_shift(c)), 4)} for c in CAT_COLS]
).sort_values("js_distance", ascending=False)
effects

**Observe:** the ranked Jensen-Shannon distances — `contact` at the top around
**0.62**, `month` around **0.55**, `poutcome` around **0.31**, everything else
(`marital`, `default`, `loan`) under **0.05**.

**Infer:** now the picture is actionable. Eleven columns "drifted" by p-value, but
only three moved by an amount a model would care about, and `contact` moved
enormously (JS distance is bounded at 1.0, so 0.62 is most of the way to two
disjoint distributions) — very likely most of Step 4's AUC drop. Had every distance
come back under 0.05 while p-values fired everywhere, the right read would be the
opposite: sample-size artifacts, raise the threshold rather than retrain. Effect
size is what turns a drift report from an alarm into a diagnosis.

## Step 7 — Target drift and prediction drift

Feature drift asks "did the inputs change?". Target drift asks "did the thing we're
predicting change?" — and prediction drift asks "did our *answers* change?". The
three are independent and each means something different.

In [ ]:
from evidently.metric_preset import TargetDriftPreset

ref_with_preds = reference.copy()
cur_with_preds = current.copy()
ref_with_preds["prediction"] = model.predict_proba(encode(reference, X_ref.columns))[:, 1]
cur_with_preds["prediction"] = cur_scores

pred_mapping = ColumnMapping(
    target="y",
    prediction="prediction",
    numerical_features=NUM_COLS,
    categorical_features=CAT_COLS,
    task="classification",
)

target_report = Report(metrics=[TargetDriftPreset()])
target_report.run(
    reference_data=ref_with_preds,
    current_data=cur_with_preds,
    column_mapping=pred_mapping,
)

td = target_report.as_dict()["metrics"][0]["result"]
for name, info in td["drift_by_columns"].items():
    print(f"{name:<12} test={info['stattest_name']:<24} "
          f"score={info['drift_score']:.5f}  drifted={info['drift_detected']}")

**Observe:** two rows. `y` drifts (chi-square p-value near **0.0**), and
`prediction` drifts too (K-S p-value near **0.0**).

**Infer:** the *combination* is what's informative — four cases worth memorising:

* **Features drift, predictions drift, target drift** (this run) — the population
  genuinely changed and the model is responding to it. Retraining is warranted.
* **Features drift, predictions stable** — the drifted features aren't ones the
  model leans on. Often safe to ignore; a good argument for weighting drift by
  feature importance rather than treating all columns equally.
* **Features stable, predictions drift** — suspicious. The inputs didn't move but
  the outputs did, which usually means a pipeline bug (a transform applied twice, a
  scaler refit) rather than real-world change.
* **Predictions stable, target drifts** — the worst case. The world moved and the
  model didn't notice at all. This is the silent failure that pure input monitoring
  can't catch, and it's why you backfill labels and recompute AUC when you can.

Note this cell needed labels for the `y` row. In production you get the
`prediction` row immediately and the `y` row much later — prediction drift is the
early warning, target drift the confirmation.

## Step 8 — Has the *explanation* drifted?

Drift in the inputs is one signal. A subtler and more interesting one: has the
model's reasoning changed? If the features it relies on to separate the classes are
different in production than they were in the reference window, the model is
effectively a different model even though the weights never changed.

Evidently doesn't compute SHAP values itself (Session 22 covers SHAP properly), but
permutation importance measured separately on each window gives the same picture
for this purpose — and unlike SHAP it needs no extra dependency.

In [ ]:
from sklearn.inspection import permutation_importance


def importance(X, y, seed=0):
    r = permutation_importance(
        model, X, y, n_repeats=5, random_state=seed,
        scoring="roc_auc", n_jobs=-1,
    )
    s = pd.Series(r.importances_mean, index=X.columns)
    # collapse one-hot columns back to their source feature
    grouped = {}
    for feat in FEATURES:
        cols = [c for c in X.columns if c == feat or c.startswith(feat + "_")]
        grouped[feat] = s[cols].sum()
    return pd.Series(grouped)


imp_ref = importance(X_val, y_val)
imp_cur = importance(X_cur, y_cur)

comparison = pd.DataFrame({
    "reference": imp_ref.round(4),
    "current": imp_cur.round(4),
})
comparison["delta"] = (comparison["current"] - comparison["reference"]).round(4)
comparison.sort_values("reference", ascending=False).head(8)

**Observe:** the top rows of the comparison. `duration` should dominate both
columns (reference importance around **0.21**, current around **0.17**). Look for
the largest *negative* deltas: `contact` and `poutcome` typically fall sharply —
`contact` from roughly **0.031** in the reference window to near **0.004** in
production.

**Infer:** `contact` carried real signal in the reference window and carries almost
none in production — exactly what Step 6's vanished `unknown` category predicts.
The model learned a split on a value that no longer occurs, so that branch is dead
weight. **Drifted *and* lost importance** is the strongest case for retraining,
because retraining will change the model's structure rather than nudge weights.

Contrast `duration`: it drifted too, but stays dominant in both windows —
drifted-but-still-important means the model is coping, so leave it. Watch for the
inverse as well: importance *rising* sharply in production often means leakage
creeping in through a changed upstream pipeline. (`duration` is itself leaky here —
call length isn't known until the call ends — kept only because it makes this
contrast legible; see "What to try next".)

## Step 9 — Turn the report into a pass/fail gate

Reports are for investigation. For a scheduled monitoring job you want a boolean.
Evidently's `TestSuite` returns exactly that, with an explicit condition per test
instead of a preset threshold buried in a banner.

In [ ]:
from evidently.test_suite import TestSuite
from evidently.tests import (
    TestShareOfDriftedColumns,
    TestColumnDrift,
    TestNumberOfMissingValues,
)

suite = TestSuite(tests=[
    TestNumberOfMissingValues(),
    TestShareOfDriftedColumns(lt=0.5),
    # the columns the model actually depends on, checked individually
    TestColumnDrift(column_name="duration"),
    TestColumnDrift(column_name="contact"),
    TestColumnDrift(column_name="poutcome"),
])

suite.run(reference_data=reference, current_data=current, column_mapping=column_mapping)

outcome = suite.as_dict()
print(f"All tests passed: {outcome['summary']['all_passed']}")
for test in outcome["tests"]:
    print(f"  [{test['status']:<7}] {test['name']}: {test['description']}")

**Observe:** `All tests passed: False`, then the per-test lines. Expect
`TestNumberOfMissingValues` to **PASS** (this dataset has no nulls — missing values
are encoded as the string `"unknown"`), `TestShareOfDriftedColumns` to **FAIL**
(0.688 is not < 0.5), and the three `TestColumnDrift` checks to fail for `contact`
and `poutcome` while `duration` fails as well on p-value grounds.

**Infer:** this is the object a monitoring job acts on — `all_passed` is your exit
code, and the `description` strings are what you post to Slack or attach to a PR
comment. Two design choices worth stealing: the per-column `TestColumnDrift` checks
exist because the aggregate share test can pass while the one column that matters
drifted catastrophically; and `TestNumberOfMissingValues` is in there because a
spike in nulls shows up as "drift" and misleads you into retraining when the real
fix is upstream. Session 11 covers that integrity side with Deepchecks; Session 17
wires this exact boolean into an automatic retraining trigger.

### When the drift report breaks or lies to you

Three failure modes, in descending order of how often they actually happen.

**1. `ImportError` after upgrading Evidently.** Version 0.7 restructured the public
API. Without a pin, Step 5 dies immediately:

```
ImportError: cannot import name 'ColumnMapping' from 'evidently'
ModuleNotFoundError: No module named 'evidently.metric_preset'
```

**Observe:** whether the traceback names `evidently.metric_preset`,
`evidently.report`, or `ColumnMapping` — all three moved in 0.7.
**Infer:** a version mismatch, not a bug in your code. Either pin back with
`pip install "evidently==0.4.40"`, or port forward: in 0.7+ `Report` comes from
`evidently` directly, presets live in `evidently.presets`, and `ColumnMapping` is
replaced by a `DataDefinition`/`Dataset` pair. Don't half-migrate — mixing import
styles produces confusing `AttributeError`s instead of a clean failure.

**2. Every column drifts.** If 16 of 16 flag, check row counts before believing it
(see Step 6), and check you didn't pass frames with mismatched dtypes — a numeric
column read as `object` in one window will always "drift".

**3. Nothing drifts, but the model degrades anyway.** Input drift detection is
blind to *concept* drift: the X→y relationship changing while X's distribution
holds still. A recession changes who repays a loan without moving the age
distribution of applicants at all. Only labels catch that — if Step 4's AUC drops
while Step 5 stays green, stop looking at feature drift and go find labels.

In [ ]:
# Guard rail: down-sample the larger window so p-value tests aren't
# swamped by sample size, then re-run the share-of-drift test
SAMPLE = 2000
ref_s = reference.sample(SAMPLE, random_state=0)
cur_s = current.sample(SAMPLE, random_state=0)

small_report = Report(metrics=[DataDriftPreset()])
small_report.run(reference_data=ref_s, current_data=cur_s, column_mapping=column_mapping)
small_summary = small_report.as_dict()["metrics"][0]["result"]

print(f"Full data   -- drifted {summary['number_of_drifted_columns']}/16 "
      f"(share {summary['share_of_drifted_columns']:.3f})")
print(f"2000-sample -- drifted {small_summary['number_of_drifted_columns']}/16 "
      f"(share {small_summary['share_of_drifted_columns']:.3f})")

**Observe:** the two lines side by side. The full-data run flags around
**11/16**; the 2,000-row sample typically flags **6/16** or **7/16** — and the
columns that survive the down-sample are `contact`, `month`, `poutcome`,
`duration`, `pdays`.

**Infer:** the columns still flagged under a smaller sample are the ones with real
effect size, and they match Step 6's Jensen-Shannon ranking almost exactly. Fixing
the comparison window size — rather than letting it grow with your warehouse — is
the simplest way to keep a monitor's sensitivity stable over time; otherwise the
alert rate climbs every month as the reference set grows and the team learns to
ignore it. If the down-sampled run flagged *more* columns, you've got the opposite
problem: the sample is too small and you're seeing noise, so raise `SAMPLE` until
the count stabilises across a few `random_state` values.

## Step 10 — Persist the artifacts

A monitoring run that leaves nothing behind is impossible to audit later. Save both
the human-readable HTML and the machine-readable JSON, keyed by the window you
scored.

In [ ]:
import json
from datetime import date

run_id = f"bank-drift-{date.today().isoformat()}"

drift_report.save_html(f"{run_id}.html")
with open(f"{run_id}.json", "w") as f:
    json.dump(drift_report.as_dict(), f, indent=2, default=str)

manifest = {
    "run_id": run_id,
    "reference_rows": len(reference),
    "current_rows": len(current),
    "drifted_columns": summary["number_of_drifted_columns"],
    "share_of_drifted_columns": round(summary["share_of_drifted_columns"], 4),
    "dataset_drift": summary["dataset_drift"],
    "reference_auc": round(float(val_auc), 4),
    "current_auc": round(float(cur_auc), 4),
    "gate_passed": outcome["summary"]["all_passed"],
}
print(json.dumps(manifest, indent=2))

**Observe:** the printed manifest — `drifted_columns: 11`,
`dataset_drift: true`, `reference_auc: 0.928`, `current_auc: 0.874`,
`gate_passed: false` — and the two files written to disk.

**Infer:** the manifest is small and flat because it's meant to be appended to a
table, one row per run, so you can plot drift share against realised AUC over time.
That time series is what tells you whether the threshold is calibrated: if
`dataset_drift` has been true for six months while AUC held steady, the threshold
is too tight and you're training the team to ignore alerts. Keeping AUC in the same
record is what makes the comparison possible — so log the drift stats immediately
and backfill the AUC field when labels land, rather than waiting for both.

## What to try next

* Retrain on the current window and re-run Step 8's importance comparison. If
  `contact` stays near zero in the retrained model, the category loss is permanent
  and the feature can be dropped from the schema entirely.
* Drop `duration` and refit. It's leaky (call length isn't known until the call
  ends), so the honest baseline AUC is meaningfully lower — around 0.79 — and the
  drift story changes shape once the dominant feature is gone.
* Session 17 wires this notebook's `gate_passed` boolean into an automated
  retraining trigger, including the promote/reject comparison this session stops
  short of.
* Session 22 replaces Step 8's permutation importance with SHAP: per-prediction
  explanations rather than per-feature averages.
* Session 11 covers the data-integrity half with Deepchecks — duplicates, nulls,
  and leakage, the failures that masquerade as drift in reports like this one.